# Fine-tuning QLoRA de Qwen2.5-7B para clasificar funciones de cita

Diseño:
- **Salida de una sola letra** (A–K = 9 funciones + `Ambiguo` + `No evaluable`). No se genera texto: se leen los logits de las 11 letras en un único *forward pass*. Eso da la **distribución de probabilidad** (confianza real, calibrable) y es mucho más rápido que generar JSON.
- Se reservan ~15 % de las anotaciones humanas como **validación** (no se usan para entrenar).
- Las clases raras se sobremuestrean hasta un mínimo (`OVERSAMPLE_MIN`).
- Opcional (`FINAL_FIT_ALL`): una vez fijados los hiperparámetros, reentrenar con las 2 700 completas.
- Después se clasifican las 18 000 citas con *checkpoint* y se exporta un CSV.

> Las etiquetas generadas siguen siendo **pre-anotación**, no gold humano. Revisa a mano las de confianza Baja. Las 2 700 anotaciones de entrenamiento las hizo el modelo haiku 5.5

## 0. Instalación (una sola vez, fuera del notebook)

```bash
cd microproyecto-local-citation/notebooks/qlora-training
uv venv .venv-qlora --python 3.12
source .venv-qlora/bin/activate

# PyTorch con CUDA (ajusta cu124 a la versión que soporte tu driver; `nvidia-smi` muestra la máxima)
uv pip install torch --index-url https://download.pytorch.org/whl/cu124
uv pip install transformers peft accelerate bitsandbytes pandas ipykernel

# Registrar el kernel y elegirlo en Jupyter/VS Code ("Python (qlora)")
python -m ipykernel install --user --name qlora --display-name "Python (qlora)"

# Comprobación
python -c "import torch, bitsandbytes; print(torch.cuda.is_available(), torch.cuda.get_device_name(0))"
```

**Libera la GPU antes de entrenar.** Ollama mantiene el modelo cargado hasta 30 min (~5-6 GB de los 12 GB):

```bash
ollama stop qwen2.5:7b-instruct      # descarga el modelo de la VRAM
# o, para parar todo el servicio:  sudo systemctl stop ollama
nvidia-smi                           # comprueba que quedan >10 GB libres
```

El modelo base (`Qwen/Qwen2.5-7B-Instruct`, ~15 GB en fp16) se descarga de Hugging Face la primera vez y se cuantiza a 4 bits al cargarlo (~5.5 GB de VRAM).

## 1. Configuración y datos

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # menos fragmentación de VRAM; debe fijarse antes de importar torch
import json, random, time
from collections import Counter
from pathlib import Path

import pandas as pd

BASE = Path('.')
GOLD_CSV   = BASE / 'ANOTACION_UNICA_2700_CITAS_REALES_ANOTADO.csv'
TARGET_CSV = BASE / 'ANOTACION_18000_CITAS_REALES.csv'

# False: entrenar con ~85 % y validar con el resto (elegir hiperparámetros).
# True: reentrenar con las 2 700 completas (modelo final); entonces la validación interna deja de ser válida.
FINAL_FIT_ALL = False

SUF = '_v1_all' if FINAL_FIT_ALL else '_v1'
OUT_CSV     = BASE / ('ANOTACION_18000_CITAS_REALES_QLORA_all.csv' if FINAL_FIT_ALL else 'ANOTACION_18000_CITAS_REALES_QLORA.csv')
ADAPTER_DIR = BASE / f'qlora_citas_adapter{SUF}'
TRAIN_DIR   = BASE / f'qlora_citas_trainer{SUF}'
SPLIT_JSON  = BASE / 'qlora_split_v1.json'
VAL_JSONL   = BASE / 'qlora_val_preds_v1.jsonl'
RUN_JSONL   = BASE / f'qlora_18000_preds{SUF}.jsonl'

BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
SEED = 42

# datos
VAL_FRAC = 0.15            # fracción de las anotaciones humanas reservada para validar
OVERSAMPLE_MIN = 100       # las clases con menos ejemplos de entrenamiento se repiten hasta este mínimo
MAX_LEN = 1024             # tokens máximos por ejemplo (se descartan los más largos)
ABSTRACT_CHARS = 400

# entrenamiento (BATCH=1 con GRAD_ACCUM=16 = lote efectivo de 16 y usa menos VRAM)
SAVE_STEPS = 25            # checkpoint cada N pasos: si se interrumpe, se pierden minutos, no horas
EPOCHS = 2
BATCH = 1
GRAD_ACCUM = 16
LR = 2e-4
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05

# inferencia
INFER_BATCH = 8
CONF_HIGH, CONF_MID = 0.80, 0.50    # umbrales de probabilidad para Alta / Media

In [2]:
FUNCTIONS = {
    'Background': 'Antecedentes generales del tema: contexto, trasfondo o historia de un campo; revisa literatura existente.',
    'Gap': 'Identifica un vacío o limitación en el trabajo previo y justifica la elección del tema.',
    'Basis': 'Fundamenta una afirmación o premisa; punto de partida sobre el que el trabajo actual construye o expande.',
    'Comparison': 'Compara explícitamente enfoques: el trabajo actual con los citados, o los citados entre sí.',
    'Application': 'Utiliza directamente, SIN modificar, un método, técnica, herramienta o dato del trabajo citado.',
    'Improvement / Modification': 'Adapta, mejora o extiende el método o herramienta citados.',
    'Evidence': 'Sustenta con evidencia o resultados: respalda afirmaciones, hipótesis o hallazgos.',
    'Identification of the Originator': 'Atribuye el origen de una idea, concepto, método o teoría a la obra pionera.',
    'Further Reading': 'Invita a leer más detalles: remite a fuentes externas para mayor profundidad.',
}
LABELS = list(FUNCTIONS) + ['Ambiguo', 'No evaluable']
LETTERS = 'ABCDEFGHIJK'[:len(LABELS)]
L2LET = dict(zip(LABELS, LETTERS)); LET2L = dict(zip(LETTERS, LABELS))
DESCR = dict(FUNCTIONS, **{'Ambiguo': 'No se puede inferir la función: hay dos o más lecturas posibles o el contexto está truncado.',
                           'No evaluable': 'El contexto no aporta ninguna información sobre la función de la cita.'})

COLS = {'ctx': 'Contexto TARGETCIT', 'title': 'Título citado', 'abs': 'Resumen citado',
        'ev': '¿Evaluable?', 'fn': 'Función principal', 'conf': 'Confianza', 'just': 'Justificación / duda'}

SYSTEM_PROMPT = ('Clasifica la función de la cita marcada TARGETCIT (OTHERCIT son otras citas) en un artículo de lingüística computacional. '
                 'Responde solo con la letra.\n'
                 + '\n'.join(f'{L2LET[k]}) {k}' for k in LABELS))

def user_message(row):
    title = row[COLS['title']].strip() or '(sin título)'
    return f"CONTEXTO:\n{row[COLS['ctx']].strip()}\n\nTRABAJO CITADO: {title}\nRESUMEN: {row[COLS['abs']].strip()[:ABSTRACT_CHARS]}"

def gold_label(row):
    return row[COLS['fn']] if row[COLS['ev']] == 'Evaluable' else row[COLS['ev']]

gold = pd.read_csv(GOLD_CSV, encoding='utf-8-sig', dtype=str, keep_default_na=False)
target = pd.read_csv(TARGET_CSV, encoding='utf-8-sig', dtype=str, keep_default_na=False)
gold['label'] = gold.apply(gold_label, axis=1)
assert set(gold['label']) <= set(LABELS), set(gold['label']) - set(LABELS)
print(len(gold), len(target)); print(gold['label'].value_counts().to_string())

2700 18000
label
Background                          1016
Application                          715
Comparison                           321
Evidence                             164
Basis                                145
Gap                                   96
Identification of the Originator      71
Improvement / Modification            58
Further Reading                       56
Ambiguo                               51
No evaluable                           7


### Partición entrenamiento / validación (por artículo citante, estratificada, guardada en disco)

Los contextos se agrupan por **artículo citante** (el prefijo de `context_id`), de modo que ningún citante queda a la vez en entrenamiento y validación. Entre muchas particiones aleatorias de grupos se elige la que deja la proporción de cada etiqueta más cercana al 15 %. La partición se guarda en `qlora_split_v1.json` para que sea **idéntica** entre ejecuciones.

También se comprueba el aislamiento frente al **Test Gold**: ningún contexto de entrenamiento puede compartir citante con él. Los artículos citados repetidos se permiten y se reportan aparte.

In [3]:
import numpy as np

gold['citing'] = gold['context_id'].map(lambda c: c.rsplit('_', 2)[0])
gold['cited']  = gold['context_id'].map(lambda c: c.rsplit('_', 2)[1])

def make_split(n_trials=3000):
    keys = sorted(gold['citing'].unique())
    classes = sorted(gold['label'].unique())
    idx = {k: i for i, k in enumerate(keys)}
    M = np.zeros((len(keys), len(classes)))
    for cit, lab in zip(gold['citing'], gold['label']):
        M[idx[cit], classes.index(lab)] += 1
    sizes = M.sum(1); tot = M.sum(0); target = VAL_FRAC * len(gold)
    best, best_score = None, np.inf
    for t in range(n_trials):
        order = np.random.default_rng(SEED + t).permutation(len(keys))
        k = int(np.searchsorted(np.cumsum(sizes[order]), target)) + 1
        val_n = M[order[:k]].sum(0)
        score = (np.abs(val_n - VAL_FRAC * tot) / tot).sum()
        if score < best_score: best, best_score = order[:k], score
    val_cit = {keys[i] for i in best}
    is_val = gold['citing'].isin(val_cit)
    return {'train': sorted(gold.loc[~is_val, 'context_id']), 'val': sorted(gold.loc[is_val, 'context_id']),
            'method': f'agrupado por citante, mejor de {n_trials} particiones, VAL_FRAC={VAL_FRAC}, SEED={SEED}'}

if SPLIT_JSON.exists():
    split = json.loads(SPLIT_JSON.read_text())
else:
    split = make_split(); SPLIT_JSON.write_text(json.dumps(split))
train_df = gold[gold['context_id'].isin(set(split['train']))]
val_df   = gold[gold['context_id'].isin(set(split['val']))]

# --- comprobaciones de aislamiento
assert not set(split['train']) & set(split['val'])
assert set(split['train']) | set(split['val']) == set(gold['context_id'])
assert not set(train_df['citing']) & set(val_df['citing']), 'un citante aparece en train y val'
val_df = val_df.assign(cited_visto=val_df['cited'].isin(set(train_df['cited'])))
print('train', len(train_df), '| val', len(val_df), '|', split.get('method', ''))
print(f"val con citado visto en train: {val_df['cited_visto'].sum()} | no visto: {(~val_df['cited_visto']).sum()}")
print(pd.DataFrame({'train': train_df['label'].value_counts(), 'val': val_df['label'].value_counts()}).fillna(0).astype(int).to_string())


def oversample(df):
    rng = random.Random(SEED); parts = [df]
    for lab, g in df.groupby('label'):
        if len(g) < OVERSAMPLE_MIN:
            parts.append(g.sample(OVERSAMPLE_MIN - len(g), replace=True, random_state=rng.randint(0, 10**6)))
    return pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)

fit_df = gold if FINAL_FIT_ALL else train_df
train_os = oversample(fit_df)
print('entrena con', 'las 2 700 completas' if FINAL_FIT_ALL else 'train', '| tras sobremuestreo:', len(train_os)); print(train_os['label'].value_counts().to_string())

train 2295 | val 405 | agrupado por citante, mejor de 3000 particiones, VAL_FRAC=0.15, SEED=42
val con citado visto en train: 263 | no visto: 142
                                  train  val
label                                       
Background                          868  148
Application                         604  111
Comparison                          273   48
Evidence                            140   24
Basis                               123   22
Gap                                  82   14
Identification of the Originator     60   11
Improvement / Modification           48   10
Further Reading                      47    9
Ambiguo                              44    7
No evaluable                          6    1
entrena con train | tras sobremuestreo: 2608
label
Background                          868
Application                         604
Comparison                          273
Evidence                            140
Basis                               123
Identification of 

## 2. Métricas (se usan para validación)

In [4]:
def report(y_true, y_pred, probs=None):
    labs = [l for l in LABELS if l in set(y_true)]
    acc = sum(a == b for a, b in zip(y_true, y_pred)) / len(y_true)
    rows = []
    for c in labs:
        tp = sum(a == b == c for a, b in zip(y_true, y_pred))
        fp = sum(b == c and a != c for a, b in zip(y_true, y_pred))
        fn = sum(a == c and b != c for a, b in zip(y_true, y_pred))
        p = tp / (tp + fp) if tp + fp else 0; r = tp / (tp + fn) if tp + fn else 0
        rows.append({'clase': c, 'soporte': tp + fn, 'predichas': tp + fp, 'precision': round(p, 3), 'recall': round(r, 3),
                     'f1': round(2 * p * r / (p + r), 3) if p + r else 0})
    rep = pd.DataFrame(rows)
    print(f'n = {len(y_true)} | exactitud = {acc:.3f} | macro-F1 = {rep.f1.mean():.3f} ')
    display(rep)
    display(pd.crosstab(pd.Series(y_true, name='humano'), pd.Series(y_pred, name='modelo')))
    if probs is not None:
        d = pd.DataFrame({'ok': [a == b for a, b in zip(y_true, y_pred)], 'p': probs})
        d['tramo'] = pd.cut(d.p, [0, .5, .8, .95, 1.0001], labels=['<0.5', '0.5-0.8', '0.8-0.95', '>0.95'])
        display(d.groupby('tramo', observed=True)['ok'].agg(['mean', 'count']).rename(columns={'mean': 'exactitud'}))
    return rep

## 3. Carga del modelo, tokenización y entrenamiento

In [5]:
import torch
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

assert torch.cuda.is_available(), 'No hay GPU CUDA visible'
print(torch.cuda.get_device_name(0), f'| VRAM libre: {torch.cuda.mem_get_info()[0] / 2**30:.1f} GiB')

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.padding_side = 'left'
if tok.pad_token is None: tok.pad_token = tok.eos_token

LETTER_IDS = [tok.encode(l, add_special_tokens=False) for l in LETTERS]
assert all(len(i) == 1 for i in LETTER_IDS), LETTER_IDS
LETTER_IDS = [i[0] for i in LETTER_IDS]
IM_END = tok.convert_tokens_to_ids('<|im_end|>')

def prompt_text(row):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user_message(row)}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def encode_example(row):
    p = tok(prompt_text(row), add_special_tokens=False)['input_ids']
    ans = [LETTER_IDS[LABELS.index(row['label'])], IM_END]
    return {'input_ids': p + ans, 'labels': [-100] * len(p) + ans}

class DS(torch.utils.data.Dataset):
    def __init__(self, df):
        ex = [encode_example(r) for _, r in df.iterrows()]
        self.items = [e for e in ex if len(e['input_ids']) <= MAX_LEN]
        print(f'{len(self.items)} ejemplos ({len(ex) - len(self.items)} descartados por > {MAX_LEN} tokens)')
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def collate(batch):
    n = max(len(b['input_ids']) for b in batch)
    ids  = torch.tensor([[tok.pad_token_id] * (n - len(b['input_ids'])) + b['input_ids'] for b in batch])
    lab  = torch.tensor([[-100] * (n - len(b['labels'])) + b['labels'] for b in batch])
    mask = torch.tensor([[0] * (n - len(b['input_ids'])) + [1] * len(b['input_ids']) for b in batch])
    return {'input_ids': ids, 'labels': lab, 'attention_mask': mask}

_ex = encode_example(train_os.iloc[0])
print('tokens del primer ejemplo:', len(_ex['input_ids']), '| respuesta:', tok.decode(_ex['input_ids'][-2:]), '=', train_os.iloc[0]['label'])

/home/carlos/microproyecto-local-citation/local_workflows/.venv-qlora/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NVIDIA GeForce RTX 4070 SUPER | VRAM libre: 10.8 GiB
tokens del primer ejemplo: 291 | respuesta: A<|im_end|> = Background


In [6]:
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map={'': 0}, torch_dtype=torch.bfloat16)
print(f'modelo base cargado | VRAM libre: {torch.cuda.mem_get_info()[0] / 2**30:.1f} GiB')

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 339/339 [00:50<00:00,  6.74it/s]


modelo base cargado | VRAM libre: 5.4 GiB


### Entrenamiento

Si `qlora_citas_adapter_v1/` ya existe se carga en lugar de reentrenar (pon `FORCE_RETRAIN = True` para repetir). Si el entrenamiento se interrumpe, se reanuda desde el último checkpoint (checkpoint cada `SAVE_STEPS` pasos) con `RESUME = True`. El adaptador queda en `qlora_citas_adapter_v1/` y los checkpoints en `qlora_citas_trainer_v1/`.

In [8]:
FORCE_RETRAIN = False
RESUME = True

if ADAPTER_DIR.exists() and not FORCE_RETRAIN and "adapter_config.json" in [f.name for f in ADAPTER_DIR.iterdir()]:
    model = PeftModel.from_pretrained(model, ADAPTER_DIR)
    print('Adaptador cargado desde', ADAPTER_DIR)
else:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                            gradient_checkpointing_kwargs={'use_reentrant': False})
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']))
    model.print_trainable_parameters()

    train_ds = DS(train_os)
    total_steps = -(-len(train_ds) // (BATCH * GRAD_ACCUM)) * EPOCHS
    args = TrainingArguments(
        output_dir=str(TRAIN_DIR), per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type='cosine', warmup_steps=max(1, int(0.05 * total_steps)),
        logging_steps=10, bf16=True, optim='paged_adamw_8bit', gradient_checkpointing=True,
        save_strategy='steps', save_steps=SAVE_STEPS, save_total_limit=2, report_to='none', remove_unused_columns=False, seed=SEED)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collate)
    has_ckpt = TRAIN_DIR.exists() and any(TRAIN_DIR.glob('checkpoint-*'))
    trainer.train(resume_from_checkpoint=True if (RESUME and has_ckpt) else None)
    model.save_pretrained(ADAPTER_DIR)
    print('Adaptador guardado en', ADAPTER_DIR)

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273
2608 ejemplos (0 descartados por > 1024 tokens)


Step,Training Loss
10,1.979753
20,1.085689
30,0.937424
40,0.846327
50,0.760445
60,0.830650
70,0.620503
80,0.584914
90,0.470118
100,0.606896


Adaptador guardado en qlora_citas_adapter_v1


## 4. Inferencia: un solo *forward pass* y probabilidades sobre las 11 letras

In [9]:
model.eval()

@torch.no_grad()
def predict_rows(rows):
    enc = tok([prompt_text(r) for r in rows], return_tensors='pt', padding=True, add_special_tokens=False).to('cuda')
    try:
        logits = model(**enc, use_cache=False, logits_to_keep=1).logits[:, -1, :]   # solo el último token: evita logits de todo el vocabulario
    except TypeError:
        logits = model(**enc, use_cache=False).logits[:, -1, :]                       # transformers antiguo: baja INFER_BATCH si hay OOM
    probs = logits[:, LETTER_IDS].float().softmax(-1).cpu()
    out = []
    for p in probs:
        k = int(p.argmax()); out.append({'label': LABELS[k], 'p': float(p[k]), 'dist': [round(float(x), 4) for x in p]})
    return out

def to_record(cid, pr):
    lab, p = pr['label'], pr['p']
    return {'context_id': cid, 'label': lab, 'p': round(p, 4), 'dist': pr['dist'],
            'evaluable': 'Evaluable' if lab in FUNCTIONS else lab,
            'funcion': lab if lab in FUNCTIONS else '',
            'confianza': 'Alta' if p >= CONF_HIGH else 'Media' if p >= CONF_MID else 'Baja'}

def load_done(path):
    done = {}
    if path.exists():
        for line in path.read_text(encoding='utf-8').splitlines():
            if line.strip():
                d = json.loads(line); done[d['context_id']] = d
    return done

def run_batch(df, path, max_rows=None):
    done = load_done(path)
    todo = df[~df['context_id'].isin(done)].copy()
    todo['_len'] = todo[COLS['ctx']].str.len() + todo[COLS['abs']].str.len().clip(upper=ABSTRACT_CHARS)
    todo = todo.sort_values('_len')                      # lotes de longitud parecida = menos padding
    if max_rows: todo = todo.head(max_rows)
    rows = [r for _, r in todo.iterrows()]
    print(f'{len(done)} ya hechos, {len(rows)} por procesar')
    t0 = time.time()
    with path.open('a', encoding='utf-8') as f:
        for i in range(0, len(rows), INFER_BATCH):
            chunk = rows[i:i + INFER_BATCH]
            for r, pr in zip(chunk, predict_rows(chunk)):
                f.write(json.dumps(to_record(r['context_id'], pr), ensure_ascii=False) + '\n')
            f.flush()
            n = i + len(chunk)
            if (i // INFER_BATCH) % 25 == 0:
                rate = n / (time.time() - t0)
                print(f'{n}/{len(rows)}  {rate:.1f} filas/s  ETA {(len(rows) - n) / rate / 60:.0f} min')
    return load_done(path)

## 5. Validación interna (solo para elegir hiperparámetros)

Esta validación está agrupada por citante, pero **no es el resultado a reportar**: el resultado final es el del Test Gold humano.

In [10]:
assert not FINAL_FIT_ALL, 'Con FINAL_FIT_ALL=True el modelo vio la validación: estas métricas no son válidas.'
vp = run_batch(val_df, VAL_JSONL)
y_true = [r['label'] for _, r in val_df.iterrows()]
y_pred = [vp[c]['label'] for c in val_df['context_id']]
probs  = [vp[c]['p'] for c in val_df['context_id']]
print('=== TODA LA VALIDACIÓN ===')
_ = report(y_true, y_pred, probs)
for nombre, mask in [('CITADO NO VISTO EN TRAIN', ~val_df['cited_visto'].values), ('CITADO VISTO EN TRAIN', val_df['cited_visto'].values)]:
    idx = np.where(mask)[0]
    print(f'=== {nombre} ({len(idx)} casos) ===')
    _ = report([y_true[i] for i in idx], [y_pred[i] for i in idx])

0 ya hechos, 405 por procesar
8/405  6.6 filas/s  ETA 1 min
208/405  7.5 filas/s  ETA 0 min
405/405  7.7 filas/s  ETA 0 min
=== TODA LA VALIDACIÓN ===
n = 405 | exactitud = 0.753 | macro-F1 = 0.528 


,clase,soporte,predichas,precision,recall,f1
0,Background,148,160,0.794,0.858,0.825
1,Gap,14,12,0.500,0.429,0.462
2,Basis,22,18,0.611,0.500,0.550
3,Comparison,48,45,0.778,0.729,0.753
4,Application,111,118,0.822,0.874,0.847
5,Improvement / Modification,10,10,0.400,0.400,0.400
6,Evidence,24,19,0.632,0.500,0.558
7,Identification of the Originator,11,13,0.538,0.636,0.583
8,Further Reading,9,7,0.714,0.556,0.625
9,Ambiguo,7,3,0.333,0.143,0.200


modelo,Ambiguo,Application,Background,Basis,Comparison,Evidence,Further Reading,Gap,Identification of the Originator,Improvement / Modification
humano,,,,,,,,,,
Ambiguo,1,1,3,0,1,0,0,0,1,0
Application,1,97,4,1,3,0,0,1,0,4
Background,0,4,127,3,2,3,2,3,3,1
Basis,0,3,4,11,1,3,0,0,0,0
Comparison,0,5,6,0,35,1,0,1,0,0
Evidence,0,2,6,1,2,12,0,1,0,0
Further Reading,1,2,0,0,0,0,5,0,0,1
Gap,0,0,5,2,0,0,0,6,1,0
Identification of the Originator,0,1,3,0,0,0,0,0,7,0


,exactitud,count
tramo,,
<0.5,0.450000,40
0.5-0.8,0.578431,102
0.8-0.95,0.775510,98
>0.95,0.921212,165


=== CITADO NO VISTO EN TRAIN (142 casos) ===
n = 142 | exactitud = 0.690 | macro-F1 = 0.509 


,clase,soporte,predichas,precision,recall,f1
0,Background,53,61,0.721,0.830,0.772
1,Gap,4,4,0.250,0.250,0.250
2,Basis,8,6,0.667,0.500,0.571
3,Comparison,14,14,0.571,0.571,0.571
4,Application,36,36,0.833,0.833,0.833
5,Improvement / Modification,6,5,0.400,0.333,0.364
6,Evidence,9,8,0.625,0.556,0.588
7,Identification of the Originator,3,4,0.500,0.667,0.571
8,Further Reading,4,3,0.667,0.500,0.571
9,Ambiguo,5,1,0.000,0.000,0.000


modelo,Ambiguo,Application,Background,Basis,Comparison,Evidence,Further Reading,Gap,Identification of the Originator,Improvement / Modification
humano,,,,,,,,,,
Ambiguo,0,0,3,0,1,0,0,0,1,0
Application,0,30,2,0,1,0,0,1,0,2
Background,0,1,44,1,2,1,1,2,1,0
Basis,0,0,2,4,1,1,0,0,0,0
Comparison,0,2,3,0,8,1,0,0,0,0
Evidence,0,1,3,0,0,5,0,0,0,0
Further Reading,1,0,0,0,0,0,2,0,0,1
Gap,0,0,2,1,0,0,0,1,0,0
Identification of the Originator,0,0,1,0,0,0,0,0,2,0


=== CITADO VISTO EN TRAIN (263 casos) ===
n = 263 | exactitud = 0.787 | macro-F1 = 0.579 


,clase,soporte,predichas,precision,recall,f1
0,Background,95,99,0.838,0.874,0.856
1,Gap,10,8,0.625,0.500,0.556
2,Basis,14,12,0.583,0.500,0.538
3,Comparison,34,31,0.871,0.794,0.831
4,Application,75,82,0.817,0.893,0.854
5,Improvement / Modification,4,5,0.400,0.500,0.444
6,Evidence,15,11,0.636,0.467,0.538
7,Identification of the Originator,8,9,0.556,0.625,0.588
8,Further Reading,5,4,0.750,0.600,0.667
9,Ambiguo,2,2,0.500,0.500,0.500


modelo,Ambiguo,Application,Background,Basis,Comparison,Evidence,Further Reading,Gap,Identification of the Originator,Improvement / Modification
humano,,,,,,,,,,
Ambiguo,1,1,0,0,0,0,0,0,0,0
Application,1,67,2,1,2,0,0,0,0,2
Background,0,3,83,2,0,2,1,1,2,1
Basis,0,3,2,7,0,2,0,0,0,0
Comparison,0,3,3,0,27,0,0,1,0,0
Evidence,0,1,3,1,2,7,0,1,0,0
Further Reading,0,2,0,0,0,0,3,0,0,0
Gap,0,0,3,1,0,0,0,5,1,0
Identification of the Originator,0,1,2,0,0,0,0,0,5,0


## 6. Clasificar las 18 000 citas

In [11]:
_ = run_batch(target, RUN_JSONL, max_rows=64)

0 ya hechos, 64 por procesar
8/64  9.0 filas/s  ETA 0 min


In [12]:
results = run_batch(target, RUN_JSONL)
print(len(results), 'de', len(target))

64 ya hechos, 17936 por procesar
8/17936  9.0 filas/s  ETA 33 min
208/17936  8.6 filas/s  ETA 34 min
408/17936  8.5 filas/s  ETA 34 min
608/17936  8.5 filas/s  ETA 34 min
808/17936  8.4 filas/s  ETA 34 min
1008/17936  8.3 filas/s  ETA 34 min
1208/17936  8.2 filas/s  ETA 34 min
1408/17936  8.1 filas/s  ETA 34 min
1608/17936  8.1 filas/s  ETA 34 min
1808/17936  8.0 filas/s  ETA 34 min
2008/17936  8.0 filas/s  ETA 33 min
2208/17936  8.0 filas/s  ETA 33 min
2408/17936  7.9 filas/s  ETA 33 min
2608/17936  7.9 filas/s  ETA 32 min
2808/17936  7.9 filas/s  ETA 32 min
3008/17936  7.9 filas/s  ETA 31 min
3208/17936  7.9 filas/s  ETA 31 min
3408/17936  7.9 filas/s  ETA 31 min
3608/17936  7.9 filas/s  ETA 30 min
3808/17936  7.9 filas/s  ETA 30 min
4008/17936  7.9 filas/s  ETA 30 min
4208/17936  7.8 filas/s  ETA 29 min
4408/17936  7.8 filas/s  ETA 29 min
4608/17936  7.8 filas/s  ETA 28 min
4808/17936  7.8 filas/s  ETA 28 min
5008/17936  7.8 filas/s  ETA 28 min
5208/17936  7.8 filas/s  ETA 27 min
54

## 7. Exportar

In [13]:
results = load_done(RUN_JSONL)
# los umbrales Alta/Media/Baja se recalculan desde la probabilidad guardada
def conf(p): return 'Alta' if p >= CONF_HIGH else 'Media' if p >= CONF_MID else 'Baja'

out = target.copy()
m = out['context_id'].map(results)
get = lambda k: m.map(lambda d: d[k] if isinstance(d, dict) else '')
out[COLS['ev']]   = get('evaluable')
out[COLS['fn']]   = get('funcion')
out[COLS['conf']] = m.map(lambda d: conf(d['p']) if isinstance(d, dict) else '')
out[COLS['just']] = m.map(lambda d: f"QLoRA p={d['p']:.2f}" if isinstance(d, dict) else '')
out['Estado']     = m.map(lambda d: ('COMPLETO' if d['evaluable'] == 'Evaluable' else 'REGISTRADO') if isinstance(d, dict) else '')
out.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')

print('filas sin clasificar:', (out[COLS['ev']] == '').sum())
print(out[COLS['ev']].value_counts().to_dict())
print(out[COLS['fn']].value_counts().to_dict())
print(out[COLS['conf']].value_counts().to_dict())
print('Filas a revisar a mano (Confianza Baja o no Evaluable):', ((out[COLS['conf']] == 'Baja') | (out[COLS['ev']] != 'Evaluable')).sum())

filas sin clasificar: 0
{'Evaluable': 17815, 'Ambiguo': 175, 'No evaluable': 10}
{'Background': 7141, 'Application': 4670, 'Comparison': 2096, 'Evidence': 1010, 'Basis': 827, 'Gap': 725, 'Identification of the Originator': 534, 'Improvement / Modification': 437, 'Further Reading': 375, '': 185}
{'Alta': 11206, 'Media': 4821, 'Baja': 1973}
Filas a revisar a mano (Confianza Baja o no Evaluable): 2023
